# Campus SVI — study area maps

Two figures describing where the campuses are and what each one looks like on the ground.
Separate from `02_analysis` because these are the only figures that touch a basemap and the only
ones drawn in geographic rather than grid space.

| Figure | Content |
|---|---|
| `fig0_study_area` | Indonesia at national extent, campus centroids with leader-line labels |
| `fig0b_campus_boundaries` | Each campus boundary over satellite imagery |

Needs only the **boundary files** — run this before acquisition finishes if you like.

---
## 1 · Setup


In [ ]:
#@title Install
!pip install -q geopandas pyogrio matplotlib contextily
print('ok')


In [ ]:
#@title Mount Drive and load the repo
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/aditpradana36/campus-svi-availability.git'  #@param {type:'string'}
PROJECT_ROOT = '/content/drive/MyDrive/campus-svi-availability'  #@param {type:'string'}

import os, sys
REPO_DIR = '/content/campus-svi-acquisition'
if not os.path.exists(REPO_DIR):
    !git clone -q $REPO_URL $REPO_DIR
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)

from campus_svi import config, boundaries, registry
config.set_root(PROJECT_ROOT)

from campus_svi.analysis import studyarea
import matplotlib.pyplot as plt

CAMPUSES = boundaries.list_campuses()
print(f'{len(CAMPUSES)} campuses'); print(', '.join(registry.display_names(CAMPUSES)))


### Institutional colours

Each campus carries its university's colour, shared across figures so a point on the overview
map and a bar in the analysis figures are recognisably the same institution. Sites of one
university share a colour — ITB Ganesha and Jatinangor, UNAIR Campus B and C, UNESA Campus 1 and
2 — because colouring them apart would imply they are different institutions.

**These are hand-entered and unverified.** Correct any you know to be wrong by editing
`INSTITUTION_COLORS` in `campus_svi/registry.py`; nothing downstream depends on the exact value.

One limitation worth stating: forty institutional colours cannot all be visually distinct, and
several universities share a similar blue or green. Colour links a point to its panel — the
labels carry identity. Never ask a reader to identify a campus from colour alone.


In [ ]:
#@title Preview the palette
import matplotlib.patches as mpatches

ids = registry.ordered(CAMPUSES)
fig, ax = plt.subplots(figsize=(7, 2.2))
for i, cid in enumerate(ids):
    ax.add_patch(mpatches.Rectangle((i % 10, -(i // 10)), 0.86, 0.86,
                                    color=registry.color(cid)))
    ax.text(i % 10 + 0.43, -(i // 10) - 0.12, registry.display_name(cid),
            ha='center', va='top', fontsize=4.2)
ax.set_xlim(-0.2, 10.2); ax.set_ylim(-((len(ids) - 1) // 10) - 0.7, 1.0)
ax.set_aspect('equal'); ax.set_axis_off(); plt.show()


---
## 2 · Overview map

Indonesia at national extent, which is strongly horizontal — hence a wide, short figure.

Labels are stacked along the left and right margins sorted by latitude, joined to their points by
leader lines, rather than placed next to each point. At 40 campuses — several of them minutes
apart in Java — any place-beside-the-point scheme collides. Stacking makes overlap impossible by
construction, and the columns are split by longitude **rank** rather than by the map's midpoint,
because Indonesian universities cluster so heavily in Java that a midpoint split leaves one
column empty.

The basemap is off by default. Satellite texture at national extent competes with forty coloured
points and says nothing about where imagery exists.


In [ ]:
#@title Figure 0 — study area overview
LABEL_PAD = 0.16  #@param {type:'number'}
POINT_SIZE = 20  #@param {type:'number'}
BASEMAP = False  #@param {type:'boolean'}

#@markdown `LABEL_PAD` is the share of width reserved for labels on each side.
#@markdown Raise it if names are clipped; lower it to give the map more room.

fig, ax = studyarea.fig_overview(CAMPUSES, label_pad=LABEL_PAD,
                                 point_size=POINT_SIZE, basemap=BASEMAP)
plt.show()


---
## 3 · Per-campus boundaries over imagery

Each campus boundary drawn over satellite imagery, with everything **outside** the boundary
under a translucent scrim and the interior left clear.

That inverts the usual highlight-the-study-area convention deliberately. The argument is about
what happens inside the fence, so the inside stays fully legible while context remains visible
enough to place the campus. The scrim is one polygon with a hole — the campus interior is
genuinely untouched, not covered by a lighter second layer.

Panels are fitted to their own campus, so scale differs between them and each carries its own
bar — the same contract as the analysis maps.

**Imagery needs network access.** If tiles fail to load, panels fall back to a plain fill and the
caption says `imagery unavailable` rather than crediting imagery that never rendered.


In [ ]:
#@title Figure 0b — campus boundaries
NCOLS = 8  #@param {type:'integer'}
SCRIM = 0.62  #@param {type:'number'}
SCRIM_COLOR = 'white'  #@param ['white', 'black']
MARGIN = 1.22  #@param {type:'number'}
IMAGERY = True  #@param {type:'boolean'}

#@markdown `SCRIM` is the opacity of the mask outside the boundary: higher
#@markdown isolates the campus more firmly, lower keeps more context.
#@markdown `MARGIN` sets how much surrounding area is in frame at all.

fig, axes = studyarea.fig_campus_panels(CAMPUSES, ncols=NCOLS, imagery=IMAGERY,
                                        scrim=SCRIM, scrim_color=SCRIM_COLOR,
                                        margin=MARGIN)
plt.show()


### Tuning the imagery

If panels look too coarse or too slow, set `zoom` explicitly. `'auto'` picks a level from the
panel extent, which for a small campus can fetch more tiles than the printed size needs. A fixed
`zoom=16` or `17` is usually enough at these panel sizes.

If tiles fail repeatedly, it is normally rate limiting rather than a code fault — wait and re-run,
or reduce the campus count for a test.


In [ ]:
#@title Both figures at once
out = studyarea.fig_study_area(CAMPUSES, ncols=NCOLS, imagery=IMAGERY)
print(f"\nwritten to {config.DATA_DIR / 'analysis' / 'figures'}")


---
## Notes

**Both figures need only boundaries**, so they can be produced before acquisition finishes.

**Colours are shared with the analysis figures** — the contributor panels, annual bars and
concentration scatter all use the same institutional colour, so a reader can carry a campus
across the figure set by eye.

**Provenance.** If you publish the imagery panels, credit Esri World Imagery and record the date
the tiles were fetched; basemap layers change without notice.
